 # MetroPT-3 — Timestamp Processing, Gap Analysis & Segmentation

**Phase 1 · Timestamp** — cleaning, chronological sort, gap detection, and segmentation.

 **What it produces** (in `OUTPUT_DIR`):
`data_ts_fixed.csv` · `timestamp_gaps.csv` · `timestamp_segments_summary.csv` · `timestamp_log.md`

###  Step 1: Imports & Centralized Configuration Management
- **Objective:** Establish a clean, maintainable configuration class using Python's `dataclasses` to manage paths, chunk sizes, and algorithmic thresholds centrally.
- **Why:** Avoids hardcoding magic numbers and paths across the pipeline, ensuring scalability, ease of maintenance, and seamless parameter tuning.

In [1]:
from __future__ import annotations
import os
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone

@dataclass
class Config:
    input_csv: Path = Path(os.environ.get(
        "TS_INPUT_CSV", "MetroPT3(AirCompressor).csv"))
    output_dir: Path = Path(os.environ.get("TS_OUTPUT_DIR", "."))
    timestamp_column: str = "timestamp"
    segment_column: str = "segment"
    drop_columns: tuple = ("Unnamed: 0",)
    chunksize: int = 200_000
    gap_multiplier: float = 3.0
    encoding: str = "utf-8-sig"

    def __post_init__(self):
        self.input_csv = Path(self.input_csv)
        self.output_dir = Path(self.output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

CFG = Config()
print(f"Configuration loaded. Input : {CFG.input_csv}")
print(f"                      Output: {CFG.output_dir.resolve()}")

Configuration loaded. Input : MetroPT3(AirCompressor).csv
                      Output: C:\Users\compu import\Desktop\TEST


In [2]:
print("Input  :", CFG.input_csv.resolve())
print("Exists :", CFG.input_csv.is_file())
print("CWD    :", Path.cwd())

Input  : C:\Users\compu import\Desktop\TEST\MetroPT3(AirCompressor).csv
Exists : True
CWD    : c:\Users\compu import\Desktop\TEST


###  Step 2: Memory-Efficient Chunked Data Ingestion
- **Objective:** Stream the large dataset in manageable batches (`chunksize=200,000`), automatically drop unwanted index artifacts, and consolidate into a unified DataFrame.
- **Why:** Prevents Out-Of-Memory (`OOM`) crashes when handling multi-million-row industrial sensor datasets.

In [3]:
def inspect_and_load(cfg: Config) -> pd.DataFrame:
    """Loads dataset efficiently and drops indexing artifacts safely."""
    if not cfg.input_csv.is_file():
        raise FileNotFoundError(f"Input file not found at: {cfg.input_csv}")
    
    print("1. Reading dataset in chunks / memory...")
    chunks = []
    for chunk in pd.read_csv(cfg.input_csv, dtype=str, keep_default_na=False, chunksize=cfg.chunksize, encoding=cfg.encoding):
        for col in cfg.drop_columns:
            if col in chunk.columns:
                chunk = chunk.drop(columns=[col])
        chunks.append(chunk)
    
    df = pd.concat(chunks, ignore_index=True)
    print(f"Loaded successfully! Total rows: {len(df):,}, Columns: {len(df.columns)}")
    return df

df = inspect_and_load(CFG)

1. Reading dataset in chunks / memory...
Loaded successfully! Total rows: 1,516,948, Columns: 16


###  Step 3: Timestamp Parsing, Chronological Sorting & Anomaly Auditing
- **Objective:** 
  1. Coerce the timestamp column into a standard datetime dtype, safely handling parsing errors into `NaT`.
  2. Sort rows chronologically to ensure strict time-series integrity.
  3. Audit and report duplicate timestamps.
- **Why:** Accurate temporal analysis depends heavily on pristine, sorted, and well-audited time indices.

In [4]:
def process_timestamps(df, cfg):
    """Parse, sort (stable), and audit duplicates WITHOUT deleting anything."""
    print("2. Parsing timestamps and sorting...")

    parsed_ts = pd.to_datetime(df[cfg.timestamp_column], errors="coerce")
    invalid_count = int(parsed_ts.isna().sum())
    if invalid_count:
        print(f"   WARNING: {invalid_count:,} invalid/missing timestamps -> excluded")

    df = df.copy()
    df[cfg.timestamp_column] = parsed_ts
    df = df.dropna(subset=[cfg.timestamp_column])
    # stable sort keeps original row order as tie-breaker
    df = df.sort_values(by=cfg.timestamp_column, kind="stable").reset_index(drop=True)

    # ---- duplicate investigation (not just a count) ----
    dup_mask = df[cfg.timestamp_column].duplicated(keep=False)
    n_dup_rows = int(dup_mask.sum())
    n_dup_groups = int(df.loc[dup_mask].groupby(cfg.timestamp_column).ngroups) if n_dup_rows else 0
    n_exact = int(df.loc[dup_mask].duplicated(keep="first").sum()) if n_dup_rows else 0

    if n_dup_rows:
        print(f"   Duplicate timestamp rows: {n_dup_rows:,} in {n_dup_groups:,} groups "
              f"({n_exact:,} exact duplicates).")
        print("   Policy: keep_all -> nothing deleted, duplicates stay visible.")

    audit = {"invalid_count": invalid_count, "dup_rows": n_dup_rows,
             "dup_groups": n_dup_groups, "exact_dups": n_exact}
    return df, audit

df, AUDIT = process_timestamps(df, CFG)

2. Parsing timestamps and sorting...


In [ ]:
pd.set_option("display.max_columns", None)       
pd.set_option("display.width", None)      
pd.set_option("display.max_colwidth", 30)     
pd.set_option("display.float_format", "{:.4f}".format)   

In [9]:
df.head(10)
df.tail(10)

,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
1516938,2020-09-01 03:58:21,-0.0120000000000004,8.99,8.974,-0.0219999999999984,8.992,60.07499999999999,0.0449999999999999,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1516939,2020-09-01 03:58:31,-0.0120000000000004,8.974,8.96,-0.0219999999999984,8.976,60.14999999999999,0.0449999999999999,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1516940,2020-09-01 03:58:40,-0.0120000000000004,8.96,8.946,-0.0199999999999995,8.962,59.94999999999999,0.0449999999999999,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1516941,2020-09-01 03:58:50,-0.0120000000000004,8.946,8.932,-0.0219999999999984,8.948,59.85000000000001,0.0449999999999999,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1516942,2020-09-01 03:59:00,-0.0120000000000004,8.934,8.92,-0.0219999999999984,8.934,59.775000000000006,0.0449999999999999,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1516943,2020-09-01 03:59:10,-0.0139999999999993,8.918,8.906,-0.0219999999999984,8.918,59.67499999999998,0.0424999999999995,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1516944,2020-09-01 03:59:20,-0.0139999999999993,8.904,8.888,-0.0199999999999995,8.904,59.60000000000001,0.0449999999999999,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1516945,2020-09-01 03:59:30,-0.0139999999999993,8.89,8.876,-0.0219999999999984,8.892,59.60000000000001,0.0424999999999995,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1516946,2020-09-01 03:59:40,-0.0120000000000004,8.876,8.864,-0.0219999999999984,8.878,59.55,0.0449999999999999,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1516947,2020-09-01 03:59:50,-0.0139999999999993,8.86,8.848,-0.0219999999999984,8.864,59.475,0.0424999999999995,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


###  Step 4: Sampling Frequency & Interval Diagnostics
- **Objective:** Compute time deltas between consecutive records and determine the robust median sampling interval.
- **Why:** The median interval serves as the fundamental baseline metric to evaluate machine logging frequency and detect operational downtime gaps accurately.

In [ ]:
def analyze_sampling_frequency(df: pd.DataFrame, cfg: Config) -> tuple:
    """Calculates consecutive time differences and extracts robust statistical benchmarks for sampling rates."""
    print(" Analyzing sampling intervals...")
    
    dt = df[cfg.timestamp_column].diff()
    
    # Used the median to prevent long machine shutdowns or outliers from skewing the baseline
    typical_interval = dt.median()
    
    interval_stats = {
        "n_intervals": int(dt.count()),
        "median_interval": typical_interval,
        "min_interval": dt.min(),
        "max_interval": dt.max()
    }
    
    print("=" * 60)
    print(f"Typical Sampling Interval (Median): {typical_interval}")
    print(f" Min Interval: {interval_stats['min_interval']} | Max Interval: {interval_stats['max_interval']}")
    print("=" * 60)
    
    return dt, interval_stats

dt, stats = analyze_sampling_frequency(df, CFG)

⏳ Analyzing sampling intervals...
Typical Sampling Interval (Median): 0 days 00:00:10
 Min Interval: 0 days 00:00:08 | Max Interval: 2 days 00:01:58


### Step 5: Heuristic Gap Detection & Smart Continuous Segmentation
- **Objective:** Apply the 3x median rule (`gap_multiplier * typical_interval`) to identify machine operational gaps, assigning unique incrementing IDs to continuous operational blocks via cumulative summation (`cumsum`).
- **Why:** Prevents downstream machine learning models or rolling aggregations from bridging across disconnected shutdown periods, preserving true operational boundaries.

In [ ]:
def apply_segmentation(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Computes median interval, sets gap threshold, and assigns unique segment IDs."""
    print("3. Computing temporal intervals and generating segments...")
    
    dt = df[cfg.timestamp_column].diff()
    typical_interval = dt.median()
    gap_threshold = cfg.gap_multiplier * typical_interval
    
    #=======>>> Any interval greater than gap_threshold triggers a new segment<==========
    is_gap = dt > gap_threshold
    df[cfg.segment_column] = is_gap.cumsum().astype(int)
    
    print("=" * 50)
    print(f" Typical Interval (Median): {typical_interval}")
    print(f" Gap Threshold (3x Median): {gap_threshold}")
    print(f"Total Segments Created: {df[cfg.segment_column].nunique():,}")
    print("=" * 50)
    
    return df

df = apply_segmentation(df, CFG)

3. Computing temporal intervals and generating segments...
 Typical Interval (Median): 0 days 00:00:10
 Gap Threshold (3x Median): 0 days 00:00:30
Total Segments Created: 332


 ### Step 6: Schema Assertions, Gap Table, Segment Summary & Export

- **Objective:**
  1. Execute programmatic assertions verifying chronological monotonicity and valid starting indices.
  2. Build the gaps table (one row per detected gap) and the segments summary (one row per segment).
  3. Write `timestamp_log.md` with all computed metrics (row counts, duplicates, interval, gaps, segments, timezone).
  4. Export the segment-annotated dataset to disk.
- **Why:** Guarantees data integrity and produces the audited artifacts required for the downstream tasks (missing values, labeling, windowing).

In [ ]:
 def build_gap_table(df, cfg, typical_interval, gap_threshold):
    """One row per detected gap. Gap = segment id increases between two consecutive rows."""
    seg = df[cfg.segment_column].to_numpy()
    ts = df[cfg.timestamp_column].to_numpy()

    is_gap = np.zeros(len(df), dtype=bool)
    is_gap[1:] = seg[1:] > seg[:-1]
    gap_idx = np.flatnonzero(is_gap)

    cols = ["gap_id", "previous_timestamp", "next_timestamp",
            "observed_interval_s", "threshold_s",
            "reference_interval_s", "estimated_missing_samples"]
    if len(gap_idx) == 0:
        return pd.DataFrame(columns=cols)

    prev_ts = ts[gap_idx - 1]
    curr_ts = ts[gap_idx]
    observed_s = (curr_ts - prev_ts) / np.timedelta64(1, "s")
    ref_s = float(typical_interval.total_seconds())
    est_missing = np.maximum(0, np.round(observed_s / ref_s) - 1).astype(int)

    return pd.DataFrame({
        "gap_id": np.arange(len(gap_idx)),
        "previous_timestamp": prev_ts,
        "next_timestamp": curr_ts,
        "observed_interval_s": observed_s,
        "threshold_s": float(gap_threshold.total_seconds()),
        "reference_interval_s": ref_s,
        "estimated_missing_samples": est_missing,
    })[cols]


def build_segments_summary(df, cfg):
    g = df.groupby(cfg.segment_column, sort=True)[cfg.timestamp_column]
    out = pd.DataFrame({
        "start_timestamp": g.min(),
        "end_timestamp": g.max(),
        "row_count": g.size(),
    }).reset_index()
    out["duration_s"] = (out["end_timestamp"] - out["start_timestamp"]).dt.total_seconds()
    return out


def write_log(cfg, df, gaps, segments, audit, typical_interval, gap_threshold):
    ts_col = df[cfg.timestamp_column]
    if ts_col.dt.tz is not None:
        tz_note = f"timezone-aware (offset: {ts_col.dt.tz})"
    else:
        tz_note = ("timezone-naive (no offset in source). "
                   "Assumption: recorded in local wall-clock time; "
                   "the MetroPT-3 documentation does NOT specify a timezone.")
    L = [
        f"# Timestamp processing log — {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}",
        "",
        f"- Input: `{cfg.input_csv.resolve()}`",
        f"- Output dir: `{cfg.output_dir.resolve()}`",
        f"- Rows after cleaning: **{len(df):,}**",
        f"- Invalid timestamps excluded: {audit['invalid_count']:,}",
        f"- Duplicate rows (kept): {audit['dup_rows']:,} "
        f"({audit['dup_groups']:,} groups, {audit['exact_dups']:,} exact)",
        f"- Typical interval (median): {typical_interval}",
        f"- Gap threshold (3× median): {gap_threshold}",
        f"- Segments created: **{df[cfg.segment_column].nunique():,}**",
        f"- Gaps detected: **{len(gaps):,}**",
        f"- Timezone: {tz_note}",
        f"- Timestamp range: {ts_col.min()} → {ts_col.max()}",
    ]
    if len(gaps):
        total_s = gaps["observed_interval_s"].sum()
        L += [
            f"- Total observed gap time: {total_s/86400:.2f} days",
            f"- Longest gap: {gaps['observed_interval_s'].max():.1f} s",
            f"- Estimated missing samples (if regular): "
            f"{int(gaps['estimated_missing_samples'].sum()):,}",
        ]
    L += [
        "",
        "## Caveats",
        "- A gap = observed temporal gap; **not** evidence of a machine failure.",
        "- Missing-sample estimates assume regular sampling at the median interval.",
        "- Duplicates were **kept** (`keep_all`); they remain visible in the output.",
        "- Sampling rate is **empirical** (from this run), not assumed from documentation.",
    ]
    p = cfg.output_dir / "timestamp_log.md"
    p.write_text("\n".join(L) + "\n", encoding="utf-8")
    return p


 
print("4. Validating constraints and exporting...")

assert df[CFG.timestamp_column].is_monotonic_increasing, "timestamps not chronological"
assert df[CFG.segment_column].iloc[0] == 0, "segment id does not start at 0"

dt = df[CFG.timestamp_column].diff()
typical_interval = dt.median()
gap_threshold = CFG.gap_multiplier * typical_interval

gaps = build_gap_table(df, CFG, typical_interval, gap_threshold)
segments = build_segments_summary(df, CFG)

 
assert len(segments) == df[CFG.segment_column].nunique(), \
    "segment summary count != unique segment ids"
assert segments["row_count"].sum() == len(df), \
    "segment row_count sum != total rows"
if len(gaps):
    assert (gaps["observed_interval_s"] > gaps["threshold_s"]).all(), \
        "some gap has interval <= threshold"

 
data_path = CFG.output_dir / "data_ts_fixed.csv"
gaps_path = CFG.output_dir / "timestamp_gaps.csv"
seg_path  = CFG.output_dir / "timestamp_segments_summary.csv"

df.to_csv(data_path, index=False)
gaps.to_csv(gaps_path, index=False)
segments.to_csv(seg_path, index=False)
log_path = write_log(CFG, df, gaps, segments, AUDIT, typical_interval, gap_threshold)

print(" Success! Exports:")
print(f"   data     -> {data_path}")
print(f"   gaps     -> {gaps_path}  ({len(gaps):,} rows)")
print(f"   segments -> {seg_path}  ({len(segments):,} rows)")
print(f"   log      -> {log_path}")

display(df[[CFG.timestamp_column, CFG.segment_column]].head(10))

4. Validating constraints and exporting...
 Success! Exports:
   data     -> data_ts_fixed.csv
   gaps     -> timestamp_gaps.csv  (331 rows)
   segments -> timestamp_segments_summary.csv  (332 rows)
   log      -> timestamp_log.md


,timestamp,segment
0,2020-02-01 00:00:00,0
1,2020-02-01 00:00:10,0
2,2020-02-01 00:00:19,0
3,2020-02-01 00:00:29,0
4,2020-02-01 00:00:39,0
5,2020-02-01 00:00:49,0
6,2020-02-01 00:00:59,0
7,2020-02-01 00:01:09,0
8,2020-02-01 00:01:19,0
9,2020-02-01 00:01:29,0


In [16]:
import inspect
print(inspect.getsource(write_log))

def write_log(cfg, df, gaps, segments, audit, typical_interval, gap_threshold):
    ts_col = df[cfg.timestamp_column]
    if ts_col.dt.tz is not None:
        tz_note = f"timezone-aware (offset: {ts_col.dt.tz})"
    else:
        tz_note = ("timezone-naive (no offset in source). "
                   "Assumption: recorded in local wall-clock time; "
                   "the MetroPT-3 documentation does NOT specify a timezone.")
    L = [
        f"# Timestamp processing log — {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}",
        "",
        f"- Input: `{cfg.input_csv.resolve()}`",
        f"- Output dir: `{cfg.output_dir.resolve()}`",
        f"- Rows after cleaning: **{len(df):,}**",
        f"- Invalid timestamps excluded: {audit['invalid_count']:,}",
        f"- Duplicate rows (kept): {audit['dup_rows']:,} "
        f"({audit['dup_groups']:,} groups, {audit['exact_dups']:,} exact)",
        f"- Typical interval (median): {typical_interval}",
        f"- Gap threshold (3× median): 